# ROGII Anchored v13 Seq-CNN Inference (v14 code upgrade)

Inference companion to `rogii-anchored-v13-seq-trainer.ipynb`.

v14 upgrades: 4 new slope features, simplified SeqCNN (3 layers, no batchnorm),
multi-checkpoint ensemble (3 seeds), median-filter post-processing.

In [ ]:
import glob, json, hashlib, os, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.ndimage import median_filter

# Install PyTorch compatible with P100 (sm_60) GPU if needed
def _install_pytorch():
    try:
        if torch.cuda.is_available():
            cap = torch.cuda.get_device_capability(0)
            if cap[0] < 7:
                print(f'GPU sm_{cap[0]}{cap[1]} detected, installing CUDA 12.6 PyTorch...')
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'torch', 'torchvision', 'torchaudio',
                    '--index-url', 'https://download.pytorch.org/whl/cu126'])
                print('PyTorch reinstalled with CUDA 12.6 support.')
            else:
                print(f'GPU sm_{cap[0]}{cap[1]} is compatible with current PyTorch.')
    except Exception as e:
        print(f'PyTorch install check failed: {e}')

_install_pytorch()

np.random.seed(42)
torch.manual_seed(42)

CHECKPOINT_VERSION = 2
TRAINING_CONFIG = {
    'optimizer': 'adam', 'lr': 1e-3, 'weight_decay': 0.0,
    'epochs_cv': 30, 'epochs_final': 60, 'n_splits': 5, 'patience': 8,
    'seed': 42, 'feature_builder': 'anchored_v14', 'architecture': 'SeqCNN_v1',
}
ENSEMBLE_SEEDS = [42, 73, 101]

def pick_device():
    if torch.cuda.is_available():
        try:
            probe = nn.Conv1d(4, 4, 3, padding=1).to('cuda')
            _ = probe(torch.zeros(1, 4, 8, device='cuda'))
            print('cuda: usable')
            return torch.device('cuda')
        except Exception as e:
            print(f'cuda: unavailable ({e})')
    return torch.device('cpu')

device = pick_device()
print(f'device: {device}, checkpoint_version: {CHECKPOINT_VERSION}')

In [ ]:
class SeqCNN(nn.Module):
    def __init__(self, in_ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, 64, 5, padding=2), nn.ReLU(),
            nn.Conv1d(64, 64, 5, padding=2), nn.ReLU(),
            nn.Conv1d(64, 32, 5, padding=2), nn.ReLU(),
            nn.Conv1d(32, 1, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

def to_well_sequences(df_part, feats):
    out = []
    for _, g in df_part.groupby('well_id'):
        X = g[feats].fillna(0).to_numpy(np.float32)
        y = (g['TVT'] - g['tvt_anchor']).to_numpy(np.float32)
        m = (g['TVT_input_missing'] == 1).to_numpy(np.float32)
        out.append((X, y, m))
    return out

print('model definition loaded (v14: 3 layers, no batchnorm)')

In [ ]:
def _find_competition_root() -> Path:
    samples = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
    if not samples:
        raise FileNotFoundError('sample_submission.csv not found')
    return Path(sorted(samples)[-1]).parent

def _load_from_raw_train() -> pd.DataFrame:
    comp = _find_competition_root()
    train_dir = comp / 'train'
    rows = []
    for p in sorted(train_dir.glob('*__horizontal_well.csv')):
        w = p.stem.replace('__horizontal_well', '')
        d = pd.read_csv(p, low_memory=False)
        for req in ['MD', 'X', 'Y', 'Z', 'GR', 'TVT', 'TVT_input']:
            if req not in d.columns:
                d[req] = np.nan
        d['well_id'] = w
        d = d.sort_values('MD').reset_index(drop=True)
        d['GR_imp'] = d['GR'].fillna(d['GR'].median())
        if not np.isfinite(d['GR_imp']).any():
            d['GR_imp'] = 0.0
        d['GR_roll10'] = d['GR_imp'].rolling(10, min_periods=1).mean()
        d['GR_roll50'] = d['GR_imp'].rolling(50, min_periods=1).mean()
        z_fill = d['Z'].ffill().bfill().fillna(0.0).to_numpy()
        md_fill = d['MD'].ffill().bfill().fillna(1.0).to_numpy()
        d['dZ_dMD'] = np.gradient(z_fill, md_fill)
        d['TVT_input_missing'] = d['TVT_input'].isna().astype(int)
        d['TVT_input_imp'] = d['TVT_input'].fillna(d['TVT_input'].median())
        d['TVT_input_imp'] = d['TVT_input_imp'].fillna(d['TVT'].median()).fillna(0.0)
        md0 = d['MD'].iloc[0] if len(d) else 0.0
        d['cum_lateral'] = (d['MD'] - md0).clip(lower=0)
        d['well_class'] = -1
        rows.append(d[['well_id','MD','X','Y','Z','TVT','TVT_input_missing','TVT_input_imp',
                       'GR_imp','GR_roll10','GR_roll50','dZ_dMD','cum_lateral','well_class']])
    if not rows:
        raise RuntimeError('no train wells found')
    return pd.concat(rows, ignore_index=True)

def _load_test_from_raw() -> pd.DataFrame:
    comp = _find_competition_root()
    test_dir = comp / 'test'
    rows = []
    for p in sorted(test_dir.glob('*__horizontal_well.csv')):
        w = p.stem.replace('__horizontal_well', '')
        d = pd.read_csv(p, low_memory=False)
        for req in ['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input']:
            if req not in d.columns:
                d[req] = np.nan
        d['well_id'] = w
        d = d.sort_values('MD').reset_index(drop=True)
        d['GR_imp'] = d['GR'].fillna(d['GR'].median())
        if not np.isfinite(d['GR_imp']).any():
            d['GR_imp'] = 0.0
        d['GR_roll10'] = d['GR_imp'].rolling(10, min_periods=1).mean()
        d['GR_roll50'] = d['GR_imp'].rolling(50, min_periods=1).mean()
        z_fill = d['Z'].ffill().bfill().fillna(0.0).to_numpy()
        md_fill = d['MD'].ffill().bfill().fillna(1.0).to_numpy()
        d['dZ_dMD'] = np.gradient(z_fill, md_fill)
        d['TVT_input_missing'] = d['TVT_input'].isna().astype(int)
        d['TVT_input_imp'] = d['TVT_input'].fillna(d['TVT_input'].median())
        d['TVT_input_imp'] = d['TVT_input_imp'].fillna(0.0)
        d['TVT'] = d['TVT_input_imp']
        md0 = d['MD'].iloc[0] if len(d) else 0.0
        d['cum_lateral'] = (d['MD'] - md0).clip(lower=0)
        d['well_class'] = -1
        rows.append(d[['well_id','MD','X','Y','Z','TVT','TVT_input_missing','TVT_input_imp',
                       'GR_imp','GR_roll10','GR_roll50','dZ_dMD','cum_lateral','well_class']])
    if not rows:
        raise RuntimeError('no test wells found')
    return pd.concat(rows, ignore_index=True)

print('data loading utilities loaded')

In [ ]:
def build_v14(df_in: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    out = df_in.copy()
    is_obs = out['TVT_input_missing'] == 0
    grp = out.groupby('well_id')
    obs = out['TVT_input_imp'].where(is_obs)
    out['tvt_anchor'] = grp['TVT_input_imp'].transform(lambda s: obs.loc[s.index].ffill())
    end_anchor = grp['TVT_input_imp'].transform(
        lambda s: obs.loc[s.index].dropna().iloc[-1] if obs.loc[s.index].notna().any() else np.nan
    )
    out['tvt_anchor'] = out['tvt_anchor'].fillna(end_anchor)
    out['pos'] = grp.cumcount()
    pos_obs = out['pos'].where(is_obs)
    out['last_pos'] = grp['pos'].transform(lambda s: pos_obs.loc[s.index].ffill())
    out['rows_since_anchor'] = (out['pos'] - out['last_pos']).fillna(out['pos'])
    md_obs = out['MD'].where(is_obs)
    out['last_md'] = grp['MD'].transform(lambda s: md_obs.loc[s.index].ffill())
    out['md_since_anchor'] = (out['MD'] - out['last_md']).fillna(0.0)
    out['gr_d1'] = grp['GR_imp'].diff().fillna(0.0)
    out['gr_d2'] = grp['gr_d1'].diff().fillna(0.0)
    out['gr_roll10_d1'] = grp['GR_roll10'].diff().fillna(0.0)
    out['z_d1'] = grp['Z'].diff().fillna(0.0)
    out['dZ_dMD_d1'] = grp['dZ_dMD'].diff().fillna(0.0)
    z_obs = out['Z'].where(is_obs)
    out['z_at_anchor'] = grp['Z'].transform(lambda s: z_obs.loc[s.index].ffill()).fillna(out['Z'])
    out['dz_from_anchor'] = out['Z'] - out['z_at_anchor']
    tvt_obs = out['TVT_input_imp'].where(is_obs)
    out['tvt_offset_at_anchor'] = grp['TVT_input_imp'].transform(
        lambda s: tvt_obs.loc[s.index].ffill()
    ) - out['z_at_anchor']
    out['tvt_offset_at_anchor'] = out['tvt_offset_at_anchor'].fillna(out['TVT_input_imp'] - out['Z'])
    out['implied_tvt_resid_from_anchor'] = (out['Z'] + out['tvt_offset_at_anchor']) - out['tvt_anchor']
    gr_obs = out['GR_imp'].where(is_obs)
    out['tw_gr_at_anchor'] = grp['GR_imp'].transform(
        lambda s: gr_obs.loc[s.index].ffill()
    ).fillna(out['GR_imp'])
    out['delta_gr_lateral'] = out['GR_imp'] - out['tw_gr_at_anchor']
    slopes = np.full(len(out), np.nan, dtype=np.float32)
    for _, g in grp:
        idx = g.index
        local_obs = is_obs.loc[idx].values
        tvt = out.loc[idx, 'TVT_input_imp'].values
        md = out.loc[idx, 'MD'].values
        opos = np.where(local_obs)[0]
        ws = np.full(len(idx), np.nan, dtype=np.float32)
        for j in range(1, len(opos)):
            pi, ci = opos[j-1], opos[j]
            dt = tvt[ci] - tvt[pi]
            dm = md[ci] - md[pi]
            if dm > 0 and np.isnan(ws[pi]):
                ws[pi] = dt / dm
        last = 0.0
        for i in range(len(ws)):
            if not np.isnan(ws[i]): last = ws[i]
            ws[i] = last
        slopes[idx] = ws
    out['tw_slope_at_anchor'] = slopes
    out['inferred_dtvt'] = out['tw_slope_at_anchor'] * out['md_since_anchor']
    feats = [
        'cum_lateral', 'rows_since_anchor', 'md_since_anchor',
        'GR_imp', 'GR_roll10', 'GR_roll50', 'dZ_dMD',
        'gr_d1', 'gr_d2', 'gr_roll10_d1', 'z_d1', 'dZ_dMD_d1',
        'dz_from_anchor', 'tvt_offset_at_anchor', 'implied_tvt_resid_from_anchor',
        'tw_gr_at_anchor', 'tw_slope_at_anchor', 'delta_gr_lateral', 'inferred_dtvt',
    ]
    return out, feats

print('v14 feature builder loaded (19 features)')

In [ ]:
print('loading training data...')
df_train = _load_from_raw_train()
df_train = df_train.sort_values(['well_id', 'MD']).reset_index(drop=True)
print(f'train shape: {df_train.shape}, wells: {df_train["well_id"].nunique()}')
print('loading test data...')
raw_test = _load_test_from_raw()
raw_test = raw_test.sort_values(['well_id', 'MD']).reset_index(drop=True)
print(f'test shape: {raw_test.shape}, wells: {raw_test["well_id"].nunique()}')
df_train_feat, FEATS = build_v14(df_train)
test_feat, _ = build_v14(raw_test)
print(f'feature count: {len(FEATS)}')

In [ ]:
def _load_inference_checkpoints(n_ensemble=3):
    ckpts = []
    for seed in ENSEMBLE_SEEDS[:n_ensemble]:
        patterns = [f'/kaggle/input/**/checkpoint_seed{seed}.pt', f'/kaggle/input/**/seed{seed}.pt']
        found = False
        for pat in patterns:
            for p in glob.glob(pat, recursive=True):
                ckpt = torch.load(p, map_location='cpu', weights_only=False)
                ckpts.append(ckpt)
                print(f'  loaded checkpoint seed{seed} from {p}')
                found = True
                break
            if found:
                break
        if not found:
            print(f'  WARNING: no checkpoint found for seed {seed}')
    return ckpts

def _load_legacy_checkpoint():
    cands = []
    for pat in ['/kaggle/input/**/latest.pt', '/kaggle/input/**/*.pt']:
        for p in glob.glob(pat, recursive=True):
            if Path(p).is_file():
                cands.append(Path(p))
    if not cands:
        return None
    return torch.load(sorted(cands)[-1], map_location='cpu', weights_only=False)

print('loading checkpoints...')
ckpts = _load_inference_checkpoints(n_ensemble=3)
if not ckpts:
    print('no v14 checkpoints, trying v13...')
    legacy = _load_legacy_checkpoint()
    if legacy:
        ckpts = [legacy]
if not ckpts:
    print('WARNING: no checkpoints, using untrained model')
if ckpts:
    mu = np.asarray(ckpts[0].get('mu'), dtype=np.float32)
    sd = np.asarray(ckpts[0].get('sd'), dtype=np.float32)
    print(f'normalization: from checkpoint (mu shape: {mu.shape})')
else:
    mu = df_train_feat[FEATS].fillna(0).mean().to_numpy(np.float32)
    sd = df_train_feat[FEATS].fillna(0).std().to_numpy(np.float32) + 1e-6
    print('normalization: computed from train data')
test_std = test_feat.copy()
test_std[FEATS] = (test_std[FEATS].fillna(0).to_numpy(np.float32) - mu) / sd
if ckpts:
    print(f'prediction: using {len(ckpts)} checkpoint ensemble')
    pred_resid_ensemble = []
    for ckpt in ckpts:
        states = ckpt.get('model_state_dicts', [ckpt.get('model_state_dict')])
        if isinstance(states, list):
            for state in states:
                model = SeqCNN(len(FEATS)).to(device)
                model.load_state_dict(state, strict=True)
                model.eval()
                pred_resid = np.zeros(len(test_std), dtype=np.float64)
                pos = 0
                with torch.no_grad():
                    for Xw, _, _ in to_well_sequences(test_std.assign(TVT=0.0), FEATS):
                        xb = torch.from_numpy(Xw.T).unsqueeze(0).to(device)
                        p = model(xb).squeeze(0).cpu().numpy().astype(np.float64)
                        pred_resid[pos:pos+len(p)] = p
                        pos += len(p)
                pred_resid_ensemble.append(pred_resid)
                del model
        else:
            model = SeqCNN(len(FEATS)).to(device)
            model.load_state_dict(states, strict=True)
            model.eval()
            pred_resid = np.zeros(len(test_std), dtype=np.float64)
            pos = 0
            with torch.no_grad():
                for Xw, _, _ in to_well_sequences(test_std.assign(TVT=0.0), FEATS):
                    xb = torch.from_numpy(Xw.T).unsqueeze(0).to(device)
                    p = model(xb).squeeze(0).cpu().numpy().astype(np.float64)
                    pred_resid[pos:pos+len(p)] = p
                    pos += len(p)
            pred_resid_ensemble.append(pred_resid)
            del model
    pred_resid = np.mean(pred_resid_ensemble, axis=0)
else:
    print('prediction: cold start (untrained)')
    model = SeqCNN(len(FEATS)).to(device)
    model.eval()
    pred_resid = np.zeros(len(test_std), dtype=np.float64)
    pos = 0
    with torch.no_grad():
        for Xw, _, _ in to_well_sequences(test_std.assign(TVT=0.0), FEATS):
            xb = torch.from_numpy(Xw.T).unsqueeze(0).to(device)
            p = model(xb).squeeze(0).cpu().numpy().astype(np.float64)
            pred_resid[pos:pos+len(p)] = p
            pos += len(p)
obs_mask = (test_feat['TVT_input_missing'] == 0).to_numpy()
pred_resid = np.where(obs_mask, 0.0, pred_resid)
pred_tvt_cnn = test_feat['tvt_anchor'].to_numpy(dtype=np.float64) + pred_resid

def slope_anchor_fallback(grp):
    tvt_inp = grp['TVT_input_imp'].to_numpy(np.float64)
    is_obs = (grp['TVT_input_missing'] == 0).to_numpy()
    md = grp['MD'].to_numpy(np.float64)
    anchor = grp['tvt_anchor'].to_numpy(np.float64)
    pred = anchor.copy()
    obs_idx = np.where(is_obs)[0]
    if len(obs_idx) >= 2:
        i0, i1 = obs_idx[-2], obs_idx[-1]
        dx = md[i1] - md[i0]
        if dx > 0:
            slope = (tvt_inp[i1] - tvt_inp[i0]) / dx
            last_obs = obs_idx[-1]
            for i in range(last_obs + 1, len(pred)):
                pred[i] = pred[last_obs] + slope * (md[i] - md[last_obs])
    return pred

slope_pred = np.zeros(len(test_feat), dtype=np.float64)
pos = 0
for _, g in test_feat.groupby('well_id'):
    sp = slope_anchor_fallback(g)
    slope_pred[pos:pos+len(sp)] = sp
    pos += len(sp)
if ckpts:
    pred_tvt = 0.9 * pred_tvt_cnn + 0.1 * slope_pred
else:
    pred_tvt = 0.2 * pred_tvt_cnn + 0.8 * slope_pred
    print('prediction: cold start blend (0.8 slope + 0.2 CNN)')
test_feat['tvt_raw'] = pred_tvt
test_feat['tvt_smooth'] = test_feat.groupby('well_id')['tvt_raw'].transform(
    lambda x: median_filter(x.values, size=5)
)
pred_tvt = test_feat['tvt_smooth'].to_numpy()
test_feat['_idx_in_well'] = test_feat.groupby('well_id').cumcount()
test_feat['id'] = test_feat['well_id'].astype(str) + '_' + test_feat['_idx_in_well'].astype(str)
test_feat['tvt'] = pred_tvt
comp_root = _find_competition_root()
sample_sub_path = comp_root / 'sample_submission.csv'
if not sample_sub_path.exists():
    sample_sub_path = comp_root / 'competitions' / 'rogii-wellbore-geology-prediction' / 'sample_submission.csv'
ss = pd.read_csv(sample_sub_path)
sub = ss[['id']].merge(test_feat[['id', 'tvt']], on='id', how='left')
if sub['tvt'].isna().any():
    fallback = float(np.nanmedian(pred_tvt))
    n_missing = int(sub['tvt'].isna().sum())
    sub['tvt'] = sub['tvt'].fillna(fallback)
    print(f'filled {n_missing} missing ids with fallback {fallback:.3f}')
sub[['id', 'tvt']].to_csv('/kaggle/working/submission.csv', index=False)
print(f'wrote /kaggle/working/submission.csv')
print(f'submission rows: {len(sub)}, nan count: {int(sub["tvt"].isna().sum())}')
print(f'checkpoints loaded: {len(ckpts)}')
print()
print('=== Submission Sanity Check ===')
print(f'rows:       {len(sub)} (expected {len(ss)})')
print(f'nan count:  {int(sub["tvt"].isna().sum())}')
tvt_vals = sub['tvt'].to_numpy()
print(f'tvt range:  [{tvt_vals.min():.3f}, {tvt_vals.max():.3f}]')
print(f'tvt mean:   {tvt_vals.mean():.3f}')
print(f'tvt std:    {tvt_vals.std():.3f}')
print(f'ID sets match: {set(sub["id"]) == set(ss["id"])}')

In [ ]:
# Final sanity check
sub = pd.read_csv('/kaggle/working/submission.csv')
ss = pd.read_csv(sample_sub_path)
print('=== Final Sanity Check ===')
print(f'rows:       {len(sub)} (expected {len(ss)})')
print(f'nan count:  {int(sub["tvt"].isna().sum())}')
print(f'tvt range:  [{sub["tvt"].min():.3f}, {sub["tvt"].max():.3f}]')
print(f'tvt mean:   {sub["tvt"].mean():.3f}')
print(f'tvt std:    {sub["tvt"].std():.3f}')
id_match = set(sub['id'].tolist()) == set(ss['id'].tolist())
print(f'ID sets match: {id_match}')